# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202501_Fire_CA'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'landsat'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 7 .tif files in the S3 bucket.


['drcs_activations/202501_Fire_CA/landsat/LC08_colorInfrared_20250106_182824_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC08_naturalColor_20250106_182824_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC08_trueColor_20250106_182824_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC09_NBR_20250114_182831_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC09_colorInfrared_20250114_182831_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC09_naturalColor_20250114_182831_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC09_trueColor_20250114_182831_041036.tif']

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [8]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 0
  - Total size: 0.00 GB


(0, 0)

In [9]:
import re

def simple_process_files(file_list, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    print("Testing filenams:")
    for filename in file_list:
        print(f"  {rename_func(filename, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=file_list,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [10]:
keys

['drcs_activations/202501_Fire_CA/landsat/LC08_colorInfrared_20250106_182824_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC08_naturalColor_20250106_182824_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC08_trueColor_20250106_182824_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC09_NBR_20250114_182831_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC09_colorInfrared_20250114_182831_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC09_naturalColor_20250114_182831_041036.tif',
 'drcs_activations/202501_Fire_CA/landsat/LC09_trueColor_20250114_182831_041036.tif']

In [10]:
def make_regex_dict(keys, regexes, products):
    ret = {}
    for i in range(len(products)):
        matches = []
        for key in keys:
            filename = key.split("/")[-1]
            match = re.search(regexes[i], filename)
            if match is not None:
                matches.append(filename)
        if matches != []:
            ret[products[i]] = matches
    return ret

In [11]:
reg_keys = make_regex_dict(keys, [r".*_colorInfrared_.*.tif", r".*_trueColor_.*.tif", r".*_naturalColor_.*.tif", r".*_NBR_.*.tif"], ["colorInfrared", "trueColor", "naturalColor", "NBR"])

In [12]:
# Define filename creator functions for different file types

def create_cog_filename(filename, event):
    sname = filename.replace(".tif", "").split("_")
    date = datetime.strptime(sname[2], "%Y%m%d")
    new_dt_format = date.strftime("%Y-%m-%d_day")
    cog_filename = f"{event}_{sname[0]}_{sname[1]}_{sname[3]}_{sname[4]}_{new_dt_format}.tif"
    return cog_filename

print("Testing filenams:")
print(reg_keys)
for k, v in reg_keys.items():
    for file in v:
        print(create_cog_filename(file, EVENT_NAME))

Testing filenams:
{'colorInfrared': ['LC08_colorInfrared_20250106_182824_041036.tif', 'LC09_colorInfrared_20250114_182831_041036.tif'], 'trueColor': ['LC08_trueColor_20250106_182824_041036.tif', 'LC09_trueColor_20250114_182831_041036.tif'], 'naturalColor': ['LC08_naturalColor_20250106_182824_041036.tif', 'LC09_naturalColor_20250114_182831_041036.tif'], 'NBR': ['LC09_NBR_20250114_182831_041036.tif']}
202501_Fire_CA_LC08_colorInfrared_182824_041036_2025-01-06_day.tif
202501_Fire_CA_LC09_colorInfrared_182831_041036_2025-01-14_day.tif
202501_Fire_CA_LC08_trueColor_182824_041036_2025-01-06_day.tif
202501_Fire_CA_LC09_trueColor_182831_041036_2025-01-14_day.tif
202501_Fire_CA_LC08_naturalColor_182824_041036_2025-01-06_day.tif
202501_Fire_CA_LC09_naturalColor_182831_041036_2025-01-14_day.tif
202501_Fire_CA_LC09_NBR_182831_041036_2025-01-14_day.tif


In [13]:
# Process S1 WTR files
for k, v in reg_keys.items():
    results = simple_process_files(file_list = v,
                                rename_func = create_cog_filename, 
                                target_dir = f"Landsat/{k}", 
                                EVENT_NAME = EVENT_NAME)


Testing filenams:
  202501_Fire_CA_LC08_colorInfrared_182824_041036_2025-01-06_day.tif
  202501_Fire_CA_LC09_colorInfrared_182831_041036_2025-01-14_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202501_Fire_CA/landsat
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/colorInfrared

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202501_Fire_CA

[1/2] Processing: LC08_colorInfrared_20250106_182824_041036.tif
   Output filename: 202501_Fire_CA_LC08_colorInfrared_182824_041036_2025-01-06_day.tif
   [MEMORY] Initial: 294.5 MB
   [DOWNLOAD] Downloading from S3...
   [ERROR] Failed: An error occurred (404) when calling the HeadObject operation: Not Found
   ❌ Error processing LC08_colorInfrared_20250106_182824_041036.tif: An error occurred (404) when calling the HeadObject operation: Not Found

[2/2] Processing: LC09_colorInfrared_20250114_182831_041036.tif
   Output filename: 202501_Fire_CA_LC09

In [18]:
keys

['drcs_activations/202501_Fire_CA/aria/asf/Eaton.tif',
 'drcs_activations/202501_Fire_CA/aria/asf/Palisades.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250112.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250114.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250112.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250114.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track64_2025-01-09_share.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track71_2025-01-09_share.tif']

In [20]:
def create_cog_filename_k2(f, EVENT_NAME):
    """Create COG filename for water mask files with formatted date."""
    from pathlib import Path
    import re
    
    filename_stem = Path(f).stem
    
    # Find date pattern (8 digits starting with 20) at the end of the filename
    date_match = re.search(r'(20\d{6})$', filename_stem)
    
    if date_match:
        date_str = date_match.group(1)
        # Format date as YYYY-MM-DD
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}_"
        # Replace the date with the formatted date
        filename_stem = filename_stem.replace(date_str, formatted_date)
    
    cog_filename = f'{EVENT_NAME}_{filename_stem}day.tif'
    return cog_filename



filter_str = 'OPERA-DIST-ALERT-HLS'

# Test functions
print("Testing WM filename:")
filter_ = [f for f in keys if filter_str in f]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_k2(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")


Testing WM filename:
  202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_2025-01-12_day.tif
  202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_2025-01-14_day.tif
  202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_2025-01-12_day.tif
  202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_2025-01-14_day.tif


In [21]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_k2, 
                                target_dir = "HLS/aria", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_2025-01-12_day.tif
  202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_2025-01-14_day.tif
  202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_2025-01-12_day.tif
  202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_2025-01-14_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202501_Fire_CA/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/aria

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202501_Fire_CA

[1/4] Processing: drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250112.tif
   Output filename: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_2025-01-12_day.tif
   [MEMORY] Initial: 359.6 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory pe

Reading input: /tmp/tmp7eekxglz_temp.tif                   



   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-3.4028234663852886e+38, max=36.0, center sample non-zero=41513/1000000
            Estimated data coverage: 40.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpqtzgv_kz.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_2025-01-12_day.tif
   [MEMORY] Final: 417.7 MB (Change: +58.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_2025-01-12_day.tif

[2/4] Processing: drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250114.tif
   Output filename: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_2025-01-14_day.tif
   [MEMORY] Initial: 417.7 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [NODATA] Source nodata val

...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_2025-01-12_day.tif
   [MEMORY] Final: 420.7 MB (Change: +3.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_2025-01-12_day.tif

[4/4] Processing: drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250114.tif
   Output filename: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_2025-01-14_day.tif
   [MEMORY] Initial: 420.7 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [NODATA] Source

Reading input: /tmp/tmpdy34li_9_temp.tif                   



   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=426/1000000
            Estimated data coverage: 1.7% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpqf2d6n5_.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_2025-01-14_day.tif
   [MEMORY] Final: 420.7 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_2025-01-14_day.tif

✅ Batch processing complete: 4 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/aria/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/aria/files_converted.csv
📁 COGs saved locally to: output/202501_Fire_CA

📊 BATCH PROCESSING SUMMARY
Total files processed: 4
Successful: 4
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-18T18:01:58.088267


In [22]:
keys

['drcs_activations/202501_Fire_CA/aria/asf/Eaton.tif',
 'drcs_activations/202501_Fire_CA/aria/asf/Palisades.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250112.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250114.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250112.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250114.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track64_2025-01-09_share.tif',
 'drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track71_2025-01-09_share.tif']

In [24]:
def create_cog_filename_k3(f, EVENT_NAME):
    """Create COG filename for water mask files."""
    f2 = Path(f).stem
    # print(f2)
    
    f2split = f2.split('_')
    # print(f2split)
    
    cog_filename= f'{EVENT_NAME}_{f2split[0]}_{f2split[1]}_{f2split[3]}_{f2split[2]}_day.tif'
    
    # print(cog_filename)
    return cog_filename
    

filter_str = 'disturbance'

# Test functions
print("Testing WM filename:")
filter_ = [f for f in keys if filter_str in f]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_k3(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")


Testing WM filename:
  202501_Fire_CA_disturbance_track64_share_2025-01-09_day.tif
  202501_Fire_CA_disturbance_track71_share_2025-01-09_day.tif


In [25]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_k3, 
                                target_dir = "HLS/aria", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202501_Fire_CA_disturbance_track64_share_2025-01-09_day.tif
  202501_Fire_CA_disturbance_track71_share_2025-01-09_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202501_Fire_CA/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/aria

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202501_Fire_CA

[1/2] Processing: drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track64_2025-01-09_share.tif
   Output filename: 202501_Fire_CA_disturbance_track64_share_2025-01-09_day.tif
   [MEMORY] Initial: 420.7 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [NODATA] Source nodata value: 255.0
   [CHUNKS] Processing 6 chunks (3x2)
   [BAND 1/1] Processing...


Reading input: /tmp/tmpqhtjtfbj_temp.tif                 

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpv64p_25h.tif


   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=999992/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_disturbance_track64_share_2025-01-09_day.tif
   [MEMORY] Final: 420.7 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202501_Fire_CA_disturbance_tra

Reading input: /tmp/tmp8y6z83ex_temp.tif                 

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpqt09bl8g.tif


   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=999843/1000000
            Estimated data coverage: 99.9% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_disturbance_track71_share_2025-01-09_day.tif
   [MEMORY] Final: 420.7 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202501_Fire_CA_disturbance_trac

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")